# 06 · Pricing & Bundle Simulation + Retention Playbook

**Goal:** answer *"what should we change, for whom, and how much money does it save?"*

Part A: **Pricing**
1. What are our prices? (we rebuild the price list from the data)
2. Why we can't calculate an "optimal price" from this data
3. Break-even: how much must churn drop for a fiber price cut to pay off?

Part B: **What-if simulation** (using the churn model)
4. What if we give free TechSupport + OnlineSecurity? Move customers to yearly contracts? Switch them to autopay?

Part C: **Retention playbook and $ impact**
5. Give every at-risk customer the right action
6. Compare costs vs money saved (customers flagged by the model, top 10%, top 20%, everyone)

We use the **final tuned model** from notebook 05 (engineered features, tuned settings, threshold 0.35).

> Run `01_data_types.ipynb` and `05_model_improvement.ipynb` first.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression

from churn_utils import prepare as prepare_with

df = pd.read_pickle("telco_clean.pkl")
saved = joblib.load("churn_model_v2.joblib")   # final model + preparation steps from notebook 05
THRESHOLD = saved["threshold"]
print("Customers:", len(df), "| model threshold:", THRESHOLD)

# Part A · Pricing

## 1 · What are our prices?

A customer's monthly charge should be the sum of the services they have.
A **linear regression** finds the price of each service: it looks for the numbers that best add up to each customer's bill.

In [ ]:
services = ["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
            "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

has_service = pd.DataFrame()
for s in services:
    has_service[s] = (df[s] == "Yes").astype(int)
has_service["DSL internet"] = (df["InternetService"] == "DSL").astype(int)
has_service["Fiber internet"] = (df["InternetService"] == "Fiber optic").astype(int)

price_model = LinearRegression()
price_model.fit(has_service, df["MonthlyCharges"])

price_list = pd.DataFrame({"service": has_service.columns,
                           "price_per_month_$": price_model.coef_.round(2)})
print(f"How well the price list explains the bills (R²): {price_model.score(has_service, df['MonthlyCharges']):.4f}")
price_list.sort_values("price_per_month_$", ascending=False)

**Finding:** R² is about 0.999, so the bills are almost exactly a **fixed price list**:

| Service | Price / month |
|---|---|
| Fiber internet | **$50** |
| DSL internet | $25 |
| Phone | $20 |
| Streaming TV / Movies | $10 each |
| Each other add-on (security, backup, protection, tech support, extra lines) | $5 each |

**Fiber costs twice as much as DSL**, and in notebook 04 fiber was the biggest driver of churn (about 3.5x the odds).

## 2 · Why we can't compute an "optimal price"

Two customers with the same package always pay the same price. We never see the same package sold at different prices,
so the data **cannot tell us** how customers react to a price change. Any "optimal price" would be a guess.

What we *can* do honestly: a **break-even analysis**.

## 3 · Break-even: when does a fiber price cut pay off?

If we cut the fiber price by $X for all fiber customers:
- **Cost:** every fiber customer pays $X less per month
- **Gain:** fewer fiber customers leave, and those we keep still pay (the new, lower price)

The question: **how many percentage points must fiber churn drop just to break even?**

In [ ]:
fiber = df[df["InternetService"] == "Fiber optic"]
fiber_customers = len(fiber)
fiber_churn = fiber["Churn"].mean()
avg_fiber_bill = fiber["MonthlyCharges"].mean()

print(f"Fiber customers: {fiber_customers} | churn: {fiber_churn * 100:.1f}% | average bill: ${avg_fiber_bill:.2f}")

rows = []
for cut in [5, 10, 15, 20]:
    yearly_cost = cut * 12 * fiber_customers * (1 - fiber_churn)   # discount paid to customers who stay
    value_of_one_saved = (avg_fiber_bill - cut) * 12                 # a saved customer pays the new price for a year
    customers_needed = yearly_cost / value_of_one_saved
    churn_drop_needed = customers_needed / fiber_customers * 100
    rows.append([f"${cut}", round(yearly_cost), round(customers_needed), round(churn_drop_needed, 1),
                 round(fiber_churn * 100 - churn_drop_needed, 1)])

breakeven = pd.DataFrame(rows, columns=["price cut / month", "yearly cost $", "customers to save",
                                        "churn must drop by (points)", "fiber churn must fall to %"])
breakeven

**How to read it:** a $10 cut only pays off if fiber churn drops by the number of points shown.
We have no proof a price cut would achieve that. In Part B we test a **cheaper alternative** the model *can* measure:
give fiber customers the add-ons that keep people (TechSupport + OnlineSecurity), worth $10/month at list price.

# Part B · What-if simulation

## Two models, two jobs

- **For ranking** (who is most at risk) we use the final notebook 05 model. It uses `class_weight="balanced"`, which pushes probabilities up.
- **For counting money** we need *realistic* probabilities, so we train the **same kind of model without balancing**.
  Its average predicted churn matches the real churn rate.

Both use the same split (`random_state=42`) and the same preparation steps (`churn_utils.py`) saved from notebook 05.

In [ ]:
rank_model = saved["model"]

def prepare(customers):
    # same steps as notebook 05 (shared code in churn_utils.py): add features, encode, impute, scale.
    # It also reminds every text column of ALL its possible values, so setting everyone's
    # Contract to "One year" in a what-if doesn't break the encoding.
    return prepare_with(customers, saved)

train, test = train_test_split(df, test_size=0.2, stratify=df["Churn"], random_state=42)

money_model = LogisticRegression(max_iter=1000)      # no balancing -> realistic probabilities
money_model.fit(prepare(train), train["Churn"])

def churn_prob(customers):
    return money_model.predict_proba(prepare(customers))[:, 1]

print(f"Real churn in test customers:      {test['Churn'].mean() * 100:.1f}%")
print(f"Predicted churn (money model):     {churn_prob(test).mean() * 100:.1f}%")
print(f"Predicted churn (ranking model):   {rank_model.predict_proba(prepare(test))[:, 1].mean() * 100:.1f}%  <- inflated")

## 4 · The actions we can simulate

Each action changes a customer's details, then we ask the model again: *"what is their churn chance now?"*

| Action | What changes | Who can get it |
|---|---|---|
| **Free add-on bundle** | TechSupport and OnlineSecurity set to "Yes" (price unchanged, it's free) | internet customers missing either |
| **Yearly contract offer** | Contract becomes "One year" | month-to-month customers |
| **Autopay switch** | Payment becomes "Credit card (automatic)" | electronic-check payers |

We simulate on the **1,409 test customers** (unseen by the model).

In [ ]:
def can_get_bundle(c):
    return (c["InternetService"] != "No") & ((c["TechSupport"] == "No") | (c["OnlineSecurity"] == "No"))

def can_get_contract(c):
    return c["Contract"] == "Month-to-month"

def can_get_autopay(c):
    return c["PaymentMethod"] == "Electronic check"

def give_bundle(c):
    c = c.copy()
    c["TechSupport"] = "Yes"
    c["OnlineSecurity"] = "Yes"
    return c

def give_contract(c):
    c = c.copy()
    c["Contract"] = "One year"
    return c

def give_autopay(c):
    c = c.copy()
    c["PaymentMethod"] = "Credit card (automatic)"
    return c

actions = {
    "Free add-on bundle": (can_get_bundle, give_bundle),
    "Yearly contract offer": (can_get_contract, give_contract),
    "Autopay switch": (can_get_autopay, give_autopay),
}

### Effect of each action on churn (if every eligible customer accepted)

In [ ]:
rows = []
for name, (eligible, apply_action) in actions.items():
    group = test[eligible(test)]
    before = churn_prob(group).mean() * 100
    after = churn_prob(apply_action(group)).mean() * 100
    rows.append([name, len(group), round(before, 1), round(after, 1), round(before - after, 1)])

effect = pd.DataFrame(rows, columns=["action", "eligible customers", "churn before %",
                                     "churn after %", "drop (points)"])
effect

In [ ]:
effect.plot(x="action", y=["churn before %", "churn after %"], kind="bar", color=["red", "green"])
plt.title("Predicted churn before vs after each action (eligible customers)")
plt.ylabel("% expected to leave")
plt.xticks(rotation=0)
plt.show()

### Compare with the fiber price cut

The bundle gives fiber customers $10/month of value. Does it reach the break-even target of a $10 price cut?

In [ ]:
fiber_test = test[(test["InternetService"] == "Fiber optic") & can_get_bundle(test)]
fiber_before = churn_prob(fiber_test).mean() * 100
fiber_after = churn_prob(give_bundle(fiber_test)).mean() * 100

needed = breakeven.loc[breakeven["price cut / month"] == "$10", "churn must drop by (points)"].iloc[0]
print(f"Fiber customers missing add-ons: churn {fiber_before:.1f}% -> {fiber_after:.1f}% with the bundle "
      f"(drop of {fiber_before - fiber_after:.1f} points)")
print(f"A $10 price cut needs a drop of {needed} points just to break even, with no evidence it would happen.")

# Part C · Retention playbook and $ impact

## 5 · Assumptions (change these to test other scenarios)

These are **business assumptions**, not data. We state them openly so judges can challenge or change them.

In [ ]:
ACCEPT_RATE = 0.30          # share of contacted customers who accept the offer
MONTHS = 12                 # we measure money over one year

CONTRACT_DISCOUNT = 0.10    # yearly-contract offer: 10% off the monthly bill
ADDON_COST = 5              # cost to us per free add-on per month ($5 list price, a cautious guess)
AUTOPAY_CREDIT = 25         # one-time $25 bill credit for switching to autopay

## The playbook: which action for whom

Every customer gets **every action they qualify for** (a "retention package"):

| Customer looks like... | Action |
|---|---|
| Month-to-month contract | Offer a 1-year contract at 10% off |
| Has internet but no TechSupport / OnlineSecurity | Free TechSupport + OnlineSecurity for 12 months |
| Pays by electronic check | $25 credit to switch to autopay |

In [ ]:
def retention_package(customers):
    # apply every action each customer qualifies for, and add up the cost per customer
    new = customers.copy()
    cost = pd.Series(0.0, index=customers.index)
    plan = pd.Series("", index=customers.index)

    m2m = can_get_contract(customers)
    new.loc[m2m, "Contract"] = "One year"
    cost[m2m] += CONTRACT_DISCOUNT * customers.loc[m2m, "MonthlyCharges"] * MONTHS
    plan[m2m] += "Yearly contract; "

    bundle = can_get_bundle(customers)
    missing = (customers["TechSupport"] == "No").astype(int) + (customers["OnlineSecurity"] == "No").astype(int)
    new.loc[bundle, "TechSupport"] = "Yes"
    new.loc[bundle, "OnlineSecurity"] = "Yes"
    cost[bundle] += missing[bundle] * ADDON_COST * MONTHS
    plan[bundle] += "Free add-ons; "

    epay = can_get_autopay(customers)
    new.loc[epay, "PaymentMethod"] = "Credit card (automatic)"
    cost[epay] += AUTOPAY_CREDIT
    plan[epay] += "Autopay credit; "

    return new, cost, plan

## 6 · Rank the test customers and apply the playbook

- **Rank** with the final notebook 05 model: priority = churn chance x monthly bill
- **Money** with the realistic model:
  - money saved = accept rate x (churn before − churn after) x monthly bill x 12
  - cost = accept rate x cost of the package

In [ ]:
test = test.copy()
test["risk_score"] = rank_model.predict_proba(prepare(test))[:, 1]
test["priority"] = test["risk_score"] * test["MonthlyCharges"]
test = test.sort_values("priority", ascending=False)

new_test, package_cost, package_plan = retention_package(test)

test["churn_before"] = churn_prob(test)
test["churn_after"] = churn_prob(new_test)
test["plan"] = package_plan
test["saved_$"] = ACCEPT_RATE * (test["churn_before"] - test["churn_after"]) * test["MonthlyCharges"] * MONTHS
test["cost_$"] = ACCEPT_RATE * package_cost

test[["customerID", "Contract", "InternetService", "PaymentMethod", "MonthlyCharges",
      "churn_before", "churn_after", "plan", "saved_$", "cost_$"]].head(10).round(2)

## $ impact: who should we target?

Contacting more customers saves more, but also costs more. We compare four options, including
**everyone the model flags** (risk score at or above the threshold chosen in notebook 05).

In [ ]:
def impact(group, label):
    revenue_saved = group["saved_$"].sum()
    cost = group["cost_$"].sum()
    kept = (ACCEPT_RATE * (group["churn_before"] - group["churn_after"])).sum()
    return [label, len(group), round(kept, 1), round(revenue_saved), round(cost), round(revenue_saved - cost),
            round(revenue_saved / cost, 2) if cost else None]

n = len(test)
results = pd.DataFrame([
    impact(test.head(int(n * 0.10)), "Top 10% by priority"),
    impact(test.head(int(n * 0.20)), "Top 20% by priority"),
    impact(test[test["risk_score"] >= THRESHOLD], f"Flagged by model (risk >= {THRESHOLD})"),
    impact(test, "Everyone"),
], columns=["target group", "customers contacted", "customers kept",
            "revenue saved $", "cost $", "net gain $", "return per $1"])
results

### Scale up to the whole customer base

The test set is 1,409 of 7,043 customers (20%), so we multiply by 5 for the full company.

In [ ]:
scale = len(df) / len(test)
full = results.copy()
for col in ["customers contacted", "customers kept", "revenue saved $", "cost $", "net gain $"]:
    full[col] = (full[col] * scale).round()
full

In [ ]:
full.plot(x="target group", y=["revenue saved $", "cost $", "net gain $"], kind="bar",
          color=["green", "orange", "steelblue"])
plt.title("Yearly impact for the whole company")
plt.ylabel("$ per year")
plt.xticks(rotation=20, ha="right")
plt.show()

### Which action does the most work?

Same calculation, one action at a time, for the top 20%.

In [ ]:
top20 = test.head(int(n * 0.20))

rows = []
for name, (eligible, apply_action) in actions.items():
    group = top20[eligible(top20)]
    drop = churn_prob(group) - churn_prob(apply_action(group))
    revenue_saved = (ACCEPT_RATE * drop * group["MonthlyCharges"] * MONTHS).sum() * scale
    rows.append([name, len(group), round(revenue_saved)])

pd.DataFrame(rows, columns=["action", "top-20% customers eligible", "yearly revenue saved $ (whole company)"])

### What if fewer customers accept?

`ACCEPT_RATE` is our biggest assumption. Both money saved and cost scale with it, so the **return per $1 stays the same**;
only the size of the gain changes.

In [ ]:
top20_saved = top20["saved_$"].sum() / ACCEPT_RATE * scale
top20_cost = top20["cost_$"].sum() / ACCEPT_RATE * scale

for rate in [0.10, 0.20, 0.30, 0.50]:
    print(f"Accept rate {rate:.0%}: net gain ${(top20_saved - top20_cost) * rate:,.0f} per year")

## Save the action list

In [ ]:
test[["customerID", "priority", "churn_before", "churn_after", "plan", "saved_$", "cost_$"]].round(3) \
    .to_csv("retention_actions.csv", index=False)
print("Saved retention_actions.csv")

## Key results

### Pricing
- Bills follow a **fixed price list** (R² = 0.999): fiber $50, DSL $25, phone $20, streaming $10, other add-ons $5.
- Nobody pays a different price for the same package, so the data **cannot** give an "optimal price". We use break-even instead.
- A **$10 fiber price cut** costs about **$216K/year** and only pays off if fiber churn drops by **7.1 points**, with no evidence it would.
- The **free TechSupport + OnlineSecurity bundle** (also $10/month of value) is predicted to cut churn for fiber customers
  missing them by **8.2 points** (47.0% → 38.8%). That's above the break-even target, and it's only given to customers who need it.

### Effect of each action (test customers, if all eligible accepted)

| Action | Eligible | Churn before | Churn after | Drop |
|---|---|---|---|---|
| Yearly contract offer | 773 | 43.0% | 29.4% | **13.6 pts** |
| Free add-on bundle | 884 | 38.1% | 31.2% | **6.9 pts** |
| Autopay switch | 474 | 45.1% | 38.5% | **6.6 pts** |

### $ impact (whole company, per year, 30% accept rate)

| Target | Contacted | Customers kept | Revenue saved | Cost | **Net gain** | Return per $1 |
|---|---|---|---|---|---|---|
| Top 10% | ~700 | ~69 | $78K | $50K | $28K | **$1.55** |
| Top 20% | ~1,405 | ~135 | $143K | $96K | **$47K** | $1.49 |
| **Flagged by model (risk ≥ 0.35)** | ~3,724 | **~266** | $240K | $200K | $40K | $1.20 |
| Everyone | 7,043 | ~282 | $251K | $236K | $15K | $1.06 |

### How to use the two lists
- The **0.35 threshold** (notebook 05) catches about 9 in 10 leavers and **keeps the most customers (about 266 a year, twice the top-20% plan)**,
  while still making money. Use it when the goal is **keeping as many customers as possible**.
- The **top 20% by priority** gives the **highest net gain** and the best return. Use it when the **budget is tight**.
- A practical mix: put every flagged customer on a **watch list with low-cost actions** (autopay nudge, onboarding call),
  and save the **expensive offers** (contract discount, free add-ons) for the top 20%.
- Contacting **everyone** is the worst option: offers go to customers who would have stayed anyway.
- In the top 20%, the **yearly contract offer** saves the most revenue, then the add-on bundle, then autopay.

### Honest limits (say these before judges ask)
- The accept rate, offer costs and add-on cost are **assumptions** (one cell at the top of Part C, easy to change).
- The model measures **association**, not proven cause. A real rollout should start with an **A/B test**.
- We only count **12 months** of revenue. A kept customer often stays for years, so the true value is likely higher.
